In [1]:


from calendar import day_name

import pandas as pd
import numpy as np
from datetime import timedelta


In [2]:
customers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_items_dataset.csv')
order_payment = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_payments_dataset.csv')
order_review = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_order_reviews_dataset.csv')
orders = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/olist_sellers_dataset.csv')
products_category_name_translation = pd.read_csv('C:/AIML/software side/olist_analisys/data/raw/product_category_name_translation.csv')


In [3]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

orders['order_delivered_customer_date'] = pd.to_datetime(
    orders['order_delivered_customer_date']
)

In [4]:
total_customer = customers['customer_unique_id'].nunique()
total_customer

96096

In [5]:
count_of_onetime = customers['customer_unique_id'].value_counts() == 1
count_of_onetime.sum()


np.int64(93099)

In [6]:
total_customer - count_of_onetime.sum()

np.int64(2997)

In [7]:
count_of_repet = customers['customer_unique_id'].value_counts().value_counts().sort_index(ascending=False)
count_of_repet

count
17        1
9         1
7         3
6         6
5         8
4        30
3       203
2      2745
1     93099
Name: count, dtype: int64

In [8]:
customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [9]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [10]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [11]:

customer_orders = customers.groupby("customer_unique_id")["customer_id"].nunique()

customer_type = customer_orders.map(
    lambda x: "One-time" if x == 1 else "Repeat"
)

customer_type = (
    customer_type.rename("customer_type")
    .reset_index()
)

customer_revenue = customers[
    ["customer_id", "customer_unique_id"]
].merge(
    orders[["order_id", "customer_id"]],
    on="customer_id",
    how="inner"
)

customer_revenue = customer_revenue.merge(
    customer_type,
    on="customer_unique_id",
    how="left"
)

customer_revenue = customer_revenue.merge(
    order_items[["order_id", "price"]],
    on="order_id",
    how="inner"
)

# Product revenue by customer type
revenue_by_type = (
    customer_revenue.groupby("customer_type")["price"]
    .sum()
    .sort_values(ascending=False)
)
display(revenue_by_type)

# Average order value by customer type
order_total = (
    customer_revenue.groupby(["order_id", "customer_type"])["price"]
    .sum()
    .reset_index()
)

order_total_mean = (
    order_total.groupby("customer_type")["price"]
    .mean()
)
display(order_total_mean)


customer_type
One-time    12812821.73
Repeat        778821.97
Name: price, dtype: float64

customer_type
One-time    138.620395
Repeat      124.911302
Name: price, dtype: float64

In [12]:
order_total = (customer_revenue.groupby(['order_id','customer_type'])['price'].sum().sort_values(ascending=False)).reset_index()
order_total

,order_id,customer_type,price
0,03caa2c082116e1d31e67e9ae3700499,One-time,13440.00
1,736e1922ae60d0d6a89247b851902527,One-time,7160.00
2,0812eb902a67711a1cb742b3cdaa65ae,One-time,6735.00
3,fefacc66af859508bf1a7934eab1e97f,One-time,6729.00
4,f5136e38d1a14a4dbd87dff67da82701,One-time,6499.00
...,...,...,...
98661,38bcb524e1c38c2c1b60600a80fc8999,Repeat,2.90
98662,e8bbc1d69fee39eee4c72cb5c969e39d,One-time,2.29
98663,f1d5c2e6867fa93ceee9ef9b34a53cbf,One-time,2.20
98664,3ee6513ae7ea23bdfab5b9ab60bffcb5,One-time,0.85


In [13]:
order_total_mean = order_total.groupby('customer_type')['price'].mean()
order_total_mean

customer_type
One-time    138.620395
Repeat      124.911302
Name: price, dtype: float64

In [14]:
customer_revenue.groupby('customer_unique_id')['price'].sum().reset_index().sort_values(by='price',ascending=False).head(10)

,customer_unique_id,price
3799,0a0a92112bd4c708ca5fde585afaa872,13440.0
81388,da122df9eeddfedc1dc1f5349a1a690c,7388.0
44139,763c8b1c9c68a0229c42c9fc6f662b93,7160.0
82230,dc4802a71eae9be1dd28f5d788ceb526,6735.0
26015,459bef486812aa25204be022145caa62,6729.0
95131,ff4159b92c40ebe40454e3e6a7c35ed6,6499.0
23947,4007669dec559734d6f53e029e360987,5934.6
89056,eebb5dda148d3893cdaf5b5ca3040ccb,4690.0
34820,5d0a2980b292d049061542014e8960bf,4599.9
27242,48e1ac109decbb87765a3eade6854098,4590.0
